# 発展：英語圏の投稿を分析する（世界情勢・国際的な話題）

Bluesky の利用者は英語圏が多数を占め，選挙・紛争・気候変動・スポーツなど **世界の出来事への反応** が英語でリアルタイムに流れています．
この Notebook では，第2〜4回で身につけた道具を英語の投稿に適用し，日本語の投稿と比べる方法を扱います．

**できるようになること**

1. 言語を指定して英語の投稿を集め，トレンドの話題を知る
2. 英語テキストの前処理（小文字化・記号除去・ストップワード）をして頻出語・特徴語を見る
3. 時差（UTC と日本時間）を踏まえて投稿の時間変化を見る
4. どの国・地域が語られているかを数える
5. 同じ話題を日本語と英語で比べる

> 英語の投稿を扱うときの注意
> - 投稿時刻は **UTC（協定世界時）** が基準です．世界中から投稿されるので，日本語の投稿のような昼夜の偏りは小さくなります．
> - 報道機関やボットの投稿（見出しの転載）が多く含まれます．投稿者（`author_handle`）の種類を確認しましょう．
> - 言語タグ `langs` は投稿者の設定に依存します．`lang="en"` で検索しても他言語が混ざることがあります．
> - 政治的・社会的に対立の激しい話題では，攻撃的な投稿も含まれます．発表では引用を最小限にし，集計で示します．

## 0. 準備

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from bsky_utils import *

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)
set_japanese_font()
login_status()

## 1. 英語で検索する

`lang="en"` を指定し，`japanese_only=False` にします（既定では日本語文字を含む投稿だけ残す設定になっているため）．

In [ ]:
posts = search_posts("climate summit", limit=20, lang="en", japanese_only=False)
print("取得件数:", len(posts))
posts[["created_at_jst", "author_handle", "text", "like_count", "langs"]].head(10)

In [ ]:
# 反応の多い順（世界的に拡散した投稿）
top = search_posts("climate summit", limit=10, lang="en", sort="top", japanese_only=False)
top[["created_at_jst", "text", "like_count", "repost_count"]]

## 2. いま何が話題か：トレンド

Bluesky が算出しているトレンド（英語圏中心）を取得できます．世界情勢の話題を探す入口になります．

In [ ]:
get_trending_topics(15)

## 3. 世界情勢の話題を2つ集めて保存する

第2回と同じ `search_posts_paged` で，比較したい2つの話題を同じ条件で集めます．ここでは「選挙（election）」と「気候（climate）」を例にします．

In [ ]:
TOPICS = ["election", "climate"]        # ← 比較したい英語の話題に書き換える（例: "ceasefire", "tariff", "AI regulation"）

frames = []
for q in TOPICS:
    print("===", q)
    frames.append(search_posts_paged(q, max_posts=300, lang="en", japanese_only=False))
df_en = pd.concat(frames, ignore_index=True)
save_posts(df_en, "data/posts_en.csv", note=f"英語圏の話題: {TOPICS}")
df_en["query"].value_counts()

In [ ]:
# 保存したデータを読み直す（API に接続できないときは合成サンプル posts_sample_en.csv が使われる）
df_en = load_posts("data/posts_en.csv", sample="posts_sample_en.csv")
df_en[["query", "created_at_jst", "author_handle", "text", "like_count"]].head()

## 4. 英語テキストの前処理

英語は空白で単語が区切られているので形態素解析は不要です．代わりに次を行います．

| 処理 | 内容 | オプション |
|---|---|---|
| 小文字化 | `Election` と `election` を同じ語にする | （常に） |
| URL・メンション除去 | 本文に残った URL や @ユーザ名を落とす | （常に） |
| 単語の切り出し | 英字の語だけ取り出す（数字・記号は落とす） | `min_len=3` |
| ハッシュタグ | `#election` のように `#` 付きで1語として残す | `keep_hashtags=True` |
| ストップワード | `stopwords_en.txt`（1行1語）＋ `extra_stopwords` | `stopwords=`, `extra_stopwords=` |

`tokenize_en()` がこれをまとめて行います．日本語と英語が混ざったデータには `tokenize_any()` を使います（日本語文字を含むかどうかで自動的に切り替え）．

In [ ]:
sample_text = "Breaking: Polls open tomorrow in the national election! Turnout expected to be the highest in decades. https://example.com #election @newsbot"
print(tokenize_en(sample_text))
print(tokenize_en(sample_text, keep_hashtags=False, extra_stopwords={"breaking"}))
print(tokenize_any("避難所の案内が分かりにくい。"), tokenize_any("The evacuation center was crowded."))

In [ ]:
extra = set(df_en["query"].unique())                       # 検索語そのものは落とす
df_en["words"] = df_en["text"].map(lambda s: tokenize_en(s, extra_stopwords=extra))
df_en[["text", "words"]].head()

## 5. 頻出語とワードクラウド

英語はそのままのフォントで描けます（日本語フォントの指定は不要）．

In [ ]:
counter = Counter(w for ws in df_en["words"] for w in ws)
labels, values = zip(*counter.most_common(25))
plt.figure(figsize=(8, 7))
plt.barh(labels[::-1], values[::-1])
plt.xlabel("count")
plt.title("Top 25 words (all posts)")
plt.tight_layout()
plt.show()

In [ ]:
from wordcloud import WordCloud

wc = WordCloud(width=900, height=500, background_color="white", colormap="viridis", max_words=100).generate_from_frequencies(counter)
plt.figure(figsize=(10, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.show()

## 6. 話題ごとの特徴語（TF-IDF）

第3回と同じ方法で，2つの話題の「らしい語」を比べます．

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

groups = df_en.groupby("query")["words"].apply(lambda s: [w for ws in s for w in ws])
vec = TfidfVectorizer(analyzer=lambda ws: ws)
X = vec.fit_transform(groups.values)
terms = np.array(vec.get_feature_names_out())

fig, axes = plt.subplots(1, len(groups), figsize=(5 * len(groups), 5))
for ax, (name, row) in zip(np.atleast_1d(axes), zip(groups.index, X.toarray())):
    idx = row.argsort()[::-1][:12]
    ax.barh(terms[idx][::-1], row[idx][::-1])
    ax.set_title(f"'{name}' (TF-IDF)")
plt.tight_layout()
plt.show()

## 7. 時差を踏まえて時間変化を見る

`created_at` は UTC です．日本語の投稿は日本時間（JST）で見ましたが，英語の投稿は世界中から来るので **UTC のまま** 見るか，関心のある地域の時刻に直します．

In [ ]:
ts_utc = pd.to_datetime(df_en["created_at"], utc=True, format="ISO8601", errors="coerce")
df_en["hour_utc"] = ts_utc.dt.hour
df_en["hour_jst"] = df_en["created_at_jst"].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, col, title in zip(axes, ["hour_utc", "hour_jst"], ["UTC", "JST（日本時間）"]):
    df_en.groupby([col, "query"]).size().unstack(fill_value=0).plot(kind="bar", ax=ax, width=0.8)
    ax.set_title(f"時刻別の投稿数（{title}）")
    ax.set_xlabel("時刻（時）")
axes[0].set_ylabel("投稿数")
plt.tight_layout()
plt.show()

In [ ]:
# 日別の推移
daily = df_en.groupby([df_en["created_at_jst"].dt.date, "query"]).size().unstack(fill_value=0)
daily.plot(figsize=(10, 4), marker="o")
plt.ylabel("投稿数")
plt.title("日別の投稿数（話題別）")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. どの国・地域が語られているか

世界情勢の分析では「どの国が話題に出るか」が基本的な指標になります．国名とその別名の小さな辞書を作り，本文に含まれる回数を数えます（辞書は自由に増やしてください）．

In [ ]:
COUNTRIES = {
    "United States": ["united states", "u.s.", "usa", "america", "american", "washington"],
    "United Kingdom": ["united kingdom", "uk", "britain", "british", "london"],
    "China": ["china", "chinese", "beijing"],
    "Japan": ["japan", "japanese", "tokyo"],
    "Russia": ["russia", "russian", "moscow"],
    "Ukraine": ["ukraine", "ukrainian", "kyiv"],
    "Germany": ["germany", "german", "berlin"],
    "France": ["france", "french", "paris"],
    "India": ["india", "indian", "delhi"],
    "Brazil": ["brazil", "brazilian"],
    "Canada": ["canada", "canadian"],
    "Australia": ["australia", "australian"],
    "Israel": ["israel", "israeli"],
    "Iran": ["iran", "iranian"],
    "South Korea": ["south korea", "korea", "korean", "seoul"],
    "Taiwan": ["taiwan", "taiwanese"],
    "Mexico": ["mexico", "mexican"],
    "EU": ["european union", " eu ", "brussels"],
    "Africa": ["africa", "african"],
    "Middle East": ["middle east", "gaza", "syria", "lebanon"],
}

def count_countries(texts):
    counts = Counter()
    for t in texts:
        low = " " + clean_text(t).lower() + " "
        for country, keys in COUNTRIES.items():
            if any(k in low for k in keys):
                counts[country] += 1
    return counts

by_topic = {q: count_countries(df_en[df_en["query"] == q]["text"]) for q in df_en["query"].unique()}
table = pd.DataFrame(by_topic).fillna(0).astype(int)
table["total"] = table.sum(axis=1)
table = table.sort_values("total", ascending=False)
table.head(15)

In [ ]:
table.drop(columns="total").head(12).plot(kind="barh", figsize=(8, 6))
plt.xlabel("言及した投稿数")
plt.title("話題別に言及された国・地域")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. 同じ話題を日本語と英語で比べる

「台風（typhoon）」のように日本と世界の両方で語られる話題を，同じ条件で集めて並べます．
語の切り出しは `tokenize_any` が言語ごとに切り替えます．

In [ ]:
PAIR = {"ja": "台風", "en": "typhoon"}          # ← 比べたい話題に書き換える（例: {"ja": "インフレ", "en": "inflation"}）

df_ja_t = search_posts_paged(PAIR["ja"], max_posts=200, lang="ja")
df_en_t = search_posts_paged(PAIR["en"], max_posts=200, lang="en", japanese_only=False)
print("日本語:", len(df_ja_t), "件  英語:", len(df_en_t), "件")

In [ ]:
pair = pd.concat([df_ja_t.assign(lang="ja"), df_en_t.assign(lang="en")], ignore_index=True)
pair["words"] = pair["text"].map(lambda s: tokenize_any(s, extra_stopwords=set(PAIR.values())))

top_words = {}
for lang in ["ja", "en"]:
    c = Counter(w for ws in pair[pair["lang"] == lang]["words"] for w in ws)
    top_words[lang] = [f"{w} ({n})" for w, n in c.most_common(15)]
pd.DataFrame(top_words)

In [ ]:
# 時刻別（UTC）の投稿数を日英で比べる：日本語は日本の昼間（UTC 0〜12時ごろ）に集中しやすい
pair["hour_utc"] = pd.to_datetime(pair["created_at"], utc=True, format="ISO8601", errors="coerce").dt.hour
pair.groupby(["hour_utc", "lang"]).size().unstack(fill_value=0).plot(kind="bar", figsize=(10, 4), width=0.8)
plt.xlabel("時刻（UTC）")
plt.ylabel("投稿数")
plt.title(f"「{PAIR['ja']}」と「{PAIR['en']}」の投稿時刻（UTC）")
plt.tight_layout()
plt.show()

In [ ]:
# 反応数の比べ方：中央値で比べる（極端な投稿に引きずられないため）
pair.groupby("lang")[["like_count", "repost_count", "reply_count"]].median()

## 10. 国際的な報道機関・機関の投稿を追う

アカウント検索で報道機関や国際機関を探し，その投稿を集めると，「公式の発信」と「一般の反応」を比べられます．

In [ ]:
actors = search_actors("world news", limit=10)
actors

In [ ]:
handle = actors.iloc[0]["handle"]          # ← 追いたいアカウントのハンドル名に書き換えてよい
news = get_author_posts(handle, max_posts=50)
print(get_profile(handle))
news[["created_at_jst", "text", "like_count"]].head(10)

## 11. 感情分析（発展）

英語の投稿には，学習済みの感情分析モデル（`cardiffnlp/twitter-roberta-base-sentiment-latest`）や，日英どちらにも使える多言語モデル（`cardiffnlp/twitter-xlm-roberta-base-sentiment`）が使えます．手順は `03_machine_learning.ipynb` の 4 節と同じです．日本語と英語を**同じ多言語モデル**で判定すると，同じ基準で比較できます．

## まとめ：英語圏の分析で使う道具

| 知りたいこと | 使う道具 |
|---|---|
| 世界で何が話題か | `get_trending_topics()`，`search_posts(..., lang="en", sort="top")` |
| ある出来事への反応を集める | `search_posts_paged("話題", lang="en", japanese_only=False)` |
| 英語の頻出語・特徴語 | `tokenize_en` → 頻出語，ワードクラウド，TF-IDF |
| いつ盛り上がったか（世界） | UTC の時刻別・日別の投稿数 |
| どの国が語られているか | 国名辞書による言及数（`COUNTRIES` を育てる） |
| 日本と世界の受け止めの違い | 同じ話題を日英で集め，`tokenize_any` で語・時刻・反応を比較 |
| 公式発信と一般の反応 | `search_actors` → `get_author_posts` |
| ポジティブ／ネガティブ | 学習済みモデル（03 の 4 節），多言語モデルで日英を同じ基準に |

**プロジェクトのヒント**：日本語だけでは投稿数が少ない国際的な話題（気候変動，紛争，国際スポーツ，技術規制など）は，英語圏のデータと組み合わせると「日本ではどう語られ，世界ではどう語られているか」という比較の問いが立てられます．